#Read csv file using data frame reader API

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config


In [0]:
%run ../00-common/02.BronzeHelper

In [0]:
catalog_name
bronze_schema
landing_folder_path


In [0]:
#source_path=landing_folder_path+"/circuits.csv"    any one can be used
source_path=f"{landing_folder_path}/{v_batch_id}/circuits.csv"
table_name=f"{catalog_name}.{bronze_schema}.circuits"

In [0]:
source_path

In [0]:

table_name

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *

circuit_schema = StructType([
  StructField('circuitId', StringType(), True),
  StructField('url', StringType(), True),
  StructField('circuitName', StringType(), True),
  StructField('lat', DoubleType(), True),
  StructField('long', DoubleType(), True),
  StructField('locality', StringType(), True),
  StructField('country', StringType(), True)
  
])

df = (spark.read.format("csv")
               .option('header', True)
             #  .option('mode', 'FAILFAST')  # strict mode datatype validation
             #  .option('mod', 'PERMISSIVE') # ignore bad records with null value
              # .option('inferSchema', True)  optional incase of schema passing as below
               .schema(circuit_schema)
               .load(source_path))

In [0]:
df.show();

In [0]:
import pyspark.sql.functions as F

df_final=add_ingestion_metadata(df)
display(df_final)

In [0]:
## df_final=df_final.withColumn("batch_id",F.lit(v_batch_id))

In [0]:
# (df_final.write
#       .format("delta")
#       .mode("overwrite")
#       .partitionBy("batch_id")
#       .option('replaceWhere',f"batch_id='{v_batch_id}'")
#       .saveAsTable(table_name))

In [0]:
write_to_bronze(input_df=df_final,
                target_table= table_name,
                batch_id= v_batch_id)

In [0]:
%sql
select * from formula1_incr_catalog.bronze.circuits

In [0]:
#spark.sql("DROP TABLE IF EXISTS formula1_incr_catalog.bronze.circuits")

In [0]:
table_name

In [0]:
df_table=spark.read.table(table_name)
display(df_table)